In [1]:
import pandas as pd
import numpy as np

In [6]:
pet_csv = "/mnt/g/My Drive/University of Connecticut/Machine Learning and Numerical Modeling in Hydrology/HBV/hbvpy/examples/fenton/MODIS16-PET_8days/MOD16A2GF-061-Statistics.csv"
temp_csv = "/mnt/g/My Drive/University of Connecticut/Machine Learning and Numerical Modeling in Hydrology/HBV/hbvpy/examples/fenton/inputPrecipTemp.csv" 

df_pet = pd.read_csv(pet_csv)

df_pet["Date"] = pd.to_datetime(df_pet["Date"], format="%m/%d/%Y", errors="coerce")
df_pet = df_pet.dropna(subset=["Date", "Mean"])

# PET 레이어만 사용
df_pet = df_pet[df_pet["Dataset"] == "PET_500m"].copy()

# feature 선택 (aid)
aids = sorted(df_pet["aid"].unique())
selected_aid = aids[0]   # 필요하면 바꿔
df_pet = df_pet[df_pet["aid"] == selected_aid].copy()

# 연, 월 추출
df_pet["year"] = df_pet["Date"].dt.year
df_pet["month"] = df_pet["Date"].dt.month

monthly_pet_by_year = (
    df_pet
    .groupby(["year", "month"], as_index=False)["Mean"]
    .sum()
    .rename(columns={"Mean": "PET_month_mm"})
)

clim_pet = (
    monthly_pet_by_year
    .groupby("month", as_index=False)["PET_month_mm"]
    .mean()
    .rename(columns={"PET_month_mm": "PEm_month"})
)



In [10]:
df_temp = pd.read_csv(temp_csv)

# Check available columns
print("Temperature CSV columns:", df_temp.columns)

# Parse datetime column
date_col = "Time"
df_temp[date_col] = pd.to_datetime(df_temp[date_col])

# Use the correct temperature column name
temp_col = "Temperature"

# Extract year and month
df_temp["year"] = df_temp[date_col].dt.year
df_temp["month"] = df_temp[date_col].dt.month

# Compute monthly mean temperature for each year
monthly_temp_by_year = (
    df_temp
    .groupby(["year", "month"], as_index=False)[temp_col]
    .mean()
)

# Compute multi-year (climatological) monthly mean temperature
clim_temp = (
    monthly_temp_by_year
    .groupby("month", as_index=False)[temp_col]
    .mean()
    .rename(columns={temp_col: "T_avg_month"})
)


Temperature CSV columns: Index(['Time', 'Month', 'Temperature', 'Precipitation'], dtype='object')


In [11]:
final = pd.merge(clim_pet, clim_temp, on="month", how="inner")

# =========================================================
# 6) 월평균 일 PET (mm/day)
# =========================================================
days_in_month = {
    1: 31, 2: 28, 3: 31, 4: 30, 5: 31, 6: 30,
    7: 31, 8: 31, 9: 30, 10: 31, 11: 30, 12: 31
}

final["PEm_day"] = final.apply(
    lambda r: r["PEm_month"] / days_in_month[int(r["month"])],
    axis=1
)

# 정리 + 반올림
final = final[["month", "T_avg_month", "PEm_month", "PEm_day"]]
final["T_avg_month"] = final["T_avg_month"].round(2)
final["PEm_month"] = final["PEm_month"].round(2)
final["PEm_day"] = final["PEm_day"].round(3)

print("\n=== Final monthly table ===")
print(final.to_string(index=False))


=== Final monthly table ===
 month  T_avg_month  PEm_month  PEm_day
     1        -2.25      20.88    0.674
     2        -1.51      34.25    1.223
     3         2.81      70.00    2.258
     4         8.48      97.47    3.249
     5        14.53     144.31    4.655
     6        19.09     185.05    6.168
     7        22.54     190.38    6.141
     8        21.47     166.38    5.367
     9        17.91     114.31    3.810
    10        11.75      50.66    1.634
    11         5.41      32.98    1.099
    12         0.56      18.88    0.609


In [12]:
out_path = "/mnt/g/My Drive/University of Connecticut/Machine Learning and Numerical Modeling in Hydrology/HBV/hbvpy/examples/fenton/inputMonthlyTempEvap.csv" 
final.to_csv(out_path, index=False)
print("\nSaved to:", out_path)


Saved to: /mnt/g/My Drive/University of Connecticut/Machine Learning and Numerical Modeling in Hydrology/HBV/hbvpy/examples/fenton/inputMonthlyTempEvap.csv
